# MaxViT — Multi-Axis Vision Transformer

**Paper:** Z. Tu, H. Talebi, H. Zhang, F. Yang, P. Milanfar, A. Bovik, and Y. Li,
"MaxViT: Multi-Axis Vision Transformer,"
in *Proc. European Conference on Computer Vision (ECCV)*, 2022, pp. 459–479.
arXiv: [2204.01697](https://arxiv.org/abs/2204.01697)

> **One-line summary:** A linear-complexity ViT that combines local block attention
> (P×P windows) and global dilated-grid attention in every block, giving full
> receptive field at O(N) cost instead of O(N²).


## Architecture Pipeline (ASCII)

```
Input Image [B, 3, 224, 224]
        │
        ▼
┌─────────────────────────────────────────┐
│  Stem: Conv(3→64, 3×3 s=2)             │  → [B, 64, 112, 112]
│         Conv(64→64, 3×3 s=1)           │
└─────────────────────┬───────────────────┘
                      │
      ┌───────────────▼───────────────┐
      │  Stage 0  [B, 64, 56, 56]    │
      │  stride-2 MBConv + N blocks  │
      │                              │
      │  Each MaxViT Block:          │
      │    MBConv (conv pre-stage)   │
      │    → Block Attn (local P×P) │
      │    → Grid Attn (global dil) │
      └───────────────┬──────────────┘
                      │
      ┌───────────────▼───────────────┐
      │  Stage 1  [B, 128, 28, 28]   │
      └───────────────┬──────────────┘
                      │
      ┌───────────────▼───────────────┐
      │  Stage 2  [B, 256, 14, 14]   │
      └───────────────┬──────────────┘
                      │
      ┌───────────────▼───────────────┐
      │  Stage 3  [B, 512,  7,  7]   │
      └───────────────┬──────────────┘
                      │
               Global Avg Pool
                      │
              pooled_output [B, 512]

──────────────────────────────────────────────────────────────────
 MaxViT Block (per block in every stage)
──────────────────────────────────────────────────────────────────

  x [B, C, H, W]  (channels-first)
       │
  MBConv  (local conv features, stride=1)
       │
  to channels-last + pad H,W → multiple of block_size P
       │
  ╔════════════════════════════════════╗
  ║  Block Attention (local)           ║
  ║  block_partition → [B*nH*nW, P², C]║
  ║  LN → MHSA + rel_bias → FFN        ║
  ║  block_reverse → [B, H, W, C]      ║
  ╚════════════════════════════════════╝
       │
  ╔════════════════════════════════════╗
  ║  Grid Attention (global dilated)   ║
  ║  grid_partition → [B*P², G², C]    ║
  ║  LN → MHSA + rel_bias → FFN        ║
  ║  grid_reverse → [B, H, W, C]       ║
  ╚════════════════════════════════════╝
       │
  unpad → back to channels-first
  x [B, C, H, W]
```


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import to_rgba
import numpy as np

# ════════════════════════════════════════════════════════════════════════════
# Figure 1: Block vs Grid partition visual on an 8×8 feature map, P=4
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

def draw_grid(ax, title, cell_colors, border_color='#455a64', window_borders=None):
    """Draw an 8×8 grid with given cell colours and optional thick window borders."""
    ax.set_xlim(-0.1, 8.1)
    ax.set_ylim(-0.1, 8.1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)

    for row in range(8):
        for col in range(8):
            color = cell_colors[row][col]
            rect = mpatches.FancyBboxPatch(
                (col + 0.04, row + 0.04), 0.92, 0.92,
                boxstyle='square,pad=0.0', facecolor=color,
                edgecolor=border_color, linewidth=0.5)
            ax.add_patch(rect)
            ax.text(col + 0.5, row + 0.5, f'{row},{col}',
                    ha='center', va='center', fontsize=5.5, color='#37474f')

    if window_borders:
        for (r0, c0, r1, c1, color, lw) in window_borders:
            rect = mpatches.Rectangle(
                (c0, r0), c1 - c0, r1 - r0,
                fill=False, edgecolor=color, linewidth=lw)
            ax.add_patch(rect)

# Colour palette for windows
W_COLORS = [
    '#bbdefb',  # W0 — top-left  block
    '#c8e6c9',  # W1 — top-right block
    '#ffe082',  # W2 — bottom-left block
    '#f8bbd0',  # W3 — bottom-right block
]
G_COLORS = [
    '#bbdefb',  # Grid 0 (top-left in each block)
    '#c8e6c9',  # Grid 1 (top-right in each block)
    '#ffe082',  # Grid 2 (bottom-left in each block)
    '#f8bbd0',  # Grid 3 (bottom-right in each block)
]

# ── Panel A: Neutral baseline (no attention)
neutral = [['#f5f5f5'] * 8 for _ in range(8)]
draw_grid(axes[0], 'Feature Map\n[8×8, block_size P=4]', neutral)
axes[0].text(4, -0.05, '64 tokens total (H×W)', ha='center', fontsize=9)

# ── Panel B: Block (local window) attention
# 4 contiguous windows of 4×4 tokens: TL, TR, BL, BR
block_colors = [[''] * 8 for _ in range(8)]
for row in range(8):
    for col in range(8):
        qr, qc = row // 4, col // 4   # which quadrant (0–3)
        win_id = qr * 2 + qc
        block_colors[row][col] = W_COLORS[win_id]

block_borders = [
    (0, 0, 4, 4, '#1565c0', 2.5),   # window 0
    (0, 4, 4, 8, '#2e7d32', 2.5),   # window 1
    (4, 0, 8, 4, '#f57f17', 2.5),   # window 2
    (4, 4, 8, 8, '#c62828', 2.5),   # window 3
]
draw_grid(axes[1], 'Block Attention\n4 local windows (P×P = 4×4)',
          block_colors, window_borders=block_borders)
axes[1].text(4, -0.05, 'Tokens in same colour attend to each other', ha='center', fontsize=8.5)

legend_b = [
    mpatches.Patch(fc='#bbdefb', ec='#1565c0', lw=2, label='Window 0 (rows 0-3, cols 0-3)'),
    mpatches.Patch(fc='#c8e6c9', ec='#2e7d32', lw=2, label='Window 1 (rows 0-3, cols 4-7)'),
    mpatches.Patch(fc='#ffe082', ec='#f57f17', lw=2, label='Window 2 (rows 4-7, cols 0-3)'),
    mpatches.Patch(fc='#f8bbd0', ec='#c62828', lw=2, label='Window 3 (rows 4-7, cols 4-7)'),
]
axes[1].legend(handles=legend_b, loc='upper center', bbox_to_anchor=(0.5, -0.18),
               fontsize=7.5, ncol=2, frameon=True)

# ── Panel C: Grid (dilated global) attention
# grid_partition with P=4: each of the 4×4=16 positions within a block forms
# a group of (8/4)×(8/4)=4 tokens spread across the whole map.
# Colour by (row%P, col%P) = local position within block.
grid_colors = [[''] * 8 for _ in range(8)]
grid_group_colors = [
    ['#bbdefb', '#aed6f1', '#5dade2', '#2e86c1'],  # row%4==0
    ['#c8e6c9', '#a9dfbf', '#52be80', '#1e8449'],  # row%4==1
    ['#ffe082', '#f9e79f', '#f4d03f', '#d4ac0d'],  # row%4==2
    ['#f8bbd0', '#f5a7bf', '#ec7fa3', '#c0392b'],  # row%4==3
]
for row in range(8):
    for col in range(8):
        grid_colors[row][col] = grid_group_colors[row % 4][col % 4]

# Overlay: mark same-group cells with a circle (all tokens with same (r%4, c%4))
draw_grid(axes[2], 'Grid Attention\n16 dilated groups (stride-P sampling)',
          grid_colors)

# Draw dotted lines connecting same-group tokens for 4 example groups
example_groups = [(0, 0), (0, 1), (1, 0), (1, 1)]
group_display_colors = ['#1565c0', '#2e7d32', '#f57f17', '#c62828']
for (gr, gc), gcolor in zip(example_groups, group_display_colors):
    pts = [(r + 0.5, c + 0.5) for r in range(8) for c in range(8) if r % 4 == gr and c % 4 == gc]
    for p in pts:
        circ = plt.Circle(p, 0.28, fill=False, ec=gcolor, lw=2.0)
        axes[2].add_patch(circ)
    for i in range(len(pts) - 1):
        axes[2].plot([pts[i][1], pts[i+1][1]], [pts[i][0], pts[i+1][0]],
                     color=gcolor, lw=1.0, linestyle='--', alpha=0.6)

axes[2].text(4, -0.05, 'Tokens with same (row%P, col%P) form one grid group', ha='center', fontsize=8)

plt.suptitle('MaxViT — Block vs Grid Attention on 8×8 Feature Map (P=4)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('maxvit_partition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Partition diagram saved.')


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# ════════════════════════════════════════════════════════════════════════════
# Figure 2: MaxViT block internals and full model architecture
# ════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(17, 12))

# ── Left: full model stages ─────────────────────────────────────────────────
ax1 = fig.add_axes([0.01, 0.05, 0.32, 0.90])
ax1.set_xlim(0, 5)
ax1.set_ylim(0, 13)
ax1.axis('off')
ax1.set_title('MaxViT-Tiny  Architecture', fontsize=12, fontweight='bold', pad=8)

model_stages = [
    ('Input [B, 3, 224, 224]',         12.4, '#e8eaf6'),
    ('Stem\nConv 3→64, s=2 + Conv 64→64\n[B, 64, 112, 112]', 11.0, '#b3e5fc'),
    ('Stage 0  depth=2\n[B, 64, 56, 56]', 9.3, '#c5cae9'),
    ('Stage 1  depth=2\n[B, 128, 28, 28]', 7.5, '#dcedc8'),
    ('Stage 2  depth=5\n[B, 256, 14, 14]', 5.6, '#fff9c4'),
    ('Stage 3  depth=2\n[B, 512,  7,  7]', 3.8, '#ffe0b2'),
    ('Global Avg Pool\npooled [B, 512]',   2.1, '#fce4ec'),
]
for label, y, color in model_stages:
    box = FancyBboxPatch((0.2, y - 0.65), 4.6, 1.1,
                         boxstyle='round,pad=0.06', facecolor=color,
                         edgecolor='#455a64', linewidth=1.8)
    ax1.add_patch(box)
    ax1.text(2.5, y, label, ha='center', va='center', fontsize=8.5)

for i in range(len(model_stages) - 1):
    y1 = model_stages[i][1] - 0.67
    y2 = model_stages[i+1][1] + 0.67
    ax1.annotate('', xy=(2.5, y2), xytext=(2.5, y1),
                 arrowprops=dict(arrowstyle='->', color='#455a64', lw=1.8))

# ── Middle: MaxViT block internals ──────────────────────────────────────────
ax2 = fig.add_axes([0.35, 0.05, 0.30, 0.90])
ax2.set_xlim(0, 4)
ax2.set_ylim(0, 14)
ax2.axis('off')
ax2.set_title('MaxViT Block Internals', fontsize=12, fontweight='bold', pad=8)

block_items = [
    ('Input x [B, C, H, W]', 13.3, '#e8eaf6'),
    ('MBConv (stride=1)\nBN → expand 4× → DW → SE → proj', 11.9, '#b3e5fc'),
    ('to channels-last\npad H,W to mult of P', 10.7, '#e0e0e0'),
    ('── Block Attention ──', 9.5, '#c5cae9'),
    ('block_partition\n[B,H,W,C]→[B*nH*nW,P²,C]', 8.4, '#c5cae9'),
    ('LN → MHSA + rel_bias → FFN', 7.3, '#bbdefb'),
    ('block_reverse → [B, H, W, C]', 6.4, '#c5cae9'),
    ('── Grid Attention ──', 5.5, '#fff9c4'),
    ('grid_partition\n[B,H,W,C]→[B*P²,G²,C]', 4.4, '#fff9c4'),
    ('LN → MHSA + dyn_rel_bias → FFN', 3.3, '#ffe082'),
    ('grid_reverse → [B, H, W, C]', 2.3, '#fff9c4'),
    ('strip pad, channels-first\n[B, C, H, W]', 1.1, '#e0e0e0'),
]
for label, y, color in block_items:
    lw = 2.0 if '──' in label else 1.1
    ec = '#1565c0' if 'Block' in label else ('#e65100' if 'Grid' in label else '#455a64')
    box = FancyBboxPatch((0.15, y - 0.47), 3.70, 0.86,
                         boxstyle='round,pad=0.04', facecolor=color,
                         edgecolor=ec, linewidth=lw)
    ax2.add_patch(box)
    ax2.text(2.0, y, label, ha='center', va='center', fontsize=8.5)

for i in range(len(block_items) - 1):
    y1 = block_items[i][1] - 0.49
    y2 = block_items[i+1][1] + 0.49
    ax2.annotate('', xy=(2.0, y2), xytext=(2.0, y1),
                 arrowprops=dict(arrowstyle='->', color='#455a64', lw=1.3))

# ── Right: WindowAttention (block vs grid) ──────────────────────────────────
ax3 = fig.add_axes([0.67, 0.42, 0.31, 0.52])
ax3.set_xlim(0, 4)
ax3.set_ylim(0, 8)
ax3.axis('off')
ax3.set_title('WindowAttention (shared class)', fontsize=10, fontweight='bold', pad=6)

attn_items = [
    ('x [B_win, N, C]', 7.4, '#e8eaf6'),
    ('QKV Linear', 6.4, '#bbdefb'),
    ('A = Q·Kᵀ / scale', 5.3, '#c5cae9'),
    ('+ rel_bias [N,N,heads]\n(dynamic if N ≠ win²)', 4.2, '#fff9c4'),
    ('Softmax', 3.2, '#e8f5e9'),
    ('· V → Proj', 2.2, '#bbdefb'),
    ('output [B_win, N, C]', 1.0, '#e8eaf6'),
]
for label, y, color in attn_items:
    box = FancyBboxPatch((0.2, y - 0.45), 3.6, 0.82,
                         boxstyle='round,pad=0.04', facecolor=color,
                         edgecolor='#455a64', linewidth=1.0)
    ax3.add_patch(box)
    ax3.text(2.0, y, label, ha='center', va='center', fontsize=9)

for i in range(len(attn_items) - 1):
    y1 = attn_items[i][1] - 0.47
    y2 = attn_items[i+1][1] + 0.47
    ax3.annotate('', xy=(2.0, y2), xytext=(2.0, y1),
                 arrowprops=dict(arrowstyle='->', color='#455a64', lw=1.2))

# ── Right bottom: N for block vs grid ───────────────────────────────────────
ax4 = fig.add_axes([0.67, 0.05, 0.31, 0.33])
ax4.set_xlim(0, 4)
ax4.set_ylim(0, 5)
ax4.axis('off')
ax4.set_title('Token Count N by Attention Type', fontsize=10, fontweight='bold', pad=6)

rows = [
    ('Block Attn', 'N = P² = 64',   'windows = (H/P)²', '#c5cae9'),
    ('Grid Attn',  'N = (H/P)² = 49', 'windows = P² = 64', '#fff9c4'),
]
ax4.text(0.1, 4.6, 'Stage 0: H=56, P=8', fontsize=9, style='italic')
for i, (typ, n_val, win_val, color) in enumerate(rows):
    y = 3.6 - i * 1.5
    box = FancyBboxPatch((0.1, y - 0.4), 3.8, 0.9,
                         boxstyle='round,pad=0.04', facecolor=color,
                         edgecolor='#455a64', linewidth=1.1)
    ax4.add_patch(box)
    ax4.text(2.0, y + 0.1, f'{typ}: {n_val}', ha='center', va='center', fontsize=9, fontweight='bold')
    ax4.text(2.0, y - 0.15, win_val, ha='center', va='center', fontsize=8)

ax4.text(0.1, 0.3,
    '⚠ Grid N ≠ P² → rel_bias index\n  rebuilt dynamically in forward()',
    fontsize=8, color='#b71c1c')

plt.suptitle('MaxViT — Full Architecture + Block Internals', fontsize=13, fontweight='bold', y=0.995)
plt.savefig('maxvit_architecture.png', dpi=140, bbox_inches='tight')
plt.show()
print('Architecture diagram saved.')


## Key Innovations

### 1. O(N) Multi-Axis Attention

Global attention has cost $O(N^2)$. MaxViT solves this with two complementary patterns:

**Block attention** (local): partition into $\left(\frac{H}{P}\right)^2$ windows of $P^2$ tokens:
$$\text{cost} = \frac{H \cdot W}{P^2} \cdot P^4 = H \cdot W \cdot P^2 = O(N \cdot P^2)$$

**Grid attention** (global): partition into $P^2$ groups of $\left(\frac{H}{P}\right)^2$ tokens:
$$\text{cost} = P^2 \cdot \left(\frac{HW}{P^2}\right)^2 \cdot \frac{1}{P^2} = \frac{H^2 W^2}{P^4}$$

Wait — but since $P$ is constant (e.g. 8) and $H, W$ scale: grid cost is $O(N^2 / P^2)$
which is *cheaper* than full attention but still $O(N)$ when $P$ is fixed.
Combined: **both heads run at linear complexity w.r.t. image size $N = H \cdot W$.**

### 2. Block Partition (O(1) reshape)

```
block_partition(x, P):  [B, H, W, C] → [B*nH*nW, P, P, C]
  nH = H//P,  nW = W//P
  x.view(B, nH, P, nW, P, C)
  .permute(0, 1, 3, 2, 4, 5)   →  [B, nH, nW, P, P, C]
  .reshape(B*nH*nW, P, P, C)
```

No data is copied — all operations are O(1) reshape + permute.

### 3. Grid Partition (O(1) reshape)

```
grid_partition(x, P):   [B, H, W, C] → [B*P*P, H//P, W//P, C]
  gh = H//P,  gw = W//P
  x.view(B, gh, P, gw, P, C)
  .permute(0, 2, 4, 1, 3, 5)   →  [B, P, P, gh, gw, C]
  .reshape(B*P*P, gh, gw, C)
```

Same reshapes — just different permute order. Grid attention picks every P-th pixel
globally, giving each window a full-image receptive field at low cost.

### 4. Dynamic Relative Bias

Block attention: window has $N = P^2$ tokens → relative bias table sized $(2P-1)^2$.

Grid attention: window has $N = (H//P)^2$ tokens → **different** size.
In our implementation, `WindowAttention.forward` detects when runtime $N \neq P^2$
and recomputes the relative position index dynamically, clamping to the pre-allocated table:

```python
win = int(N ** 0.5)
if win != self.window_size:
    idx = self._build_rel_index(win, x.device)   # dynamic computation
idx_clamped = idx.clamp(0, table_size - 1)        # stay in-bounds
bias = self.rel_bias_table[idx_clamped]           # [N², heads]
```

### 5. Padding Strategy

Both block and grid partition require $H$ and $W$ to be multiples of $P$.
For inputs where this fails (e.g. $H=7$, $P=8$), MaxViT pads before attention and
strips the pad after:

```python
pad_h = (P - H % P) % P    # 0 if already divisible
x = F.pad(x, (0,0, 0,pad_w, 0,pad_h))  # channels-last padding
# ... attention on Hp×Wp ...
x = x[:, :H, :W, :]        # strip pad
```

### 6. Model Variants

| Model | Base dim | Depths | Params |
|-------|---------|--------|--------|
| maxvit_tiny | 64 | [2,2,5,2] | ~31 M |
| maxvit_small | 96 | [2,2,5,2] | ~69 M |
| maxvit_base | 96 | [2,6,14,2] | ~120 M |
| maxvit_large | 128 | [2,6,14,2] | ~212 M |
| maxvit_xlarge | 192 | [2,6,14,2] | ~475 M |


In [ ]:
import sys
sys.path.insert(0, '../../src')
import torch
from hale_core.nn.vision import build_vision_backbone
from hale_core.nn.vision.hybrid.maxvit import block_partition, block_reverse, grid_partition, grid_reverse

# ── Verify partition/unpartition round-trips ───────────────────────────────
print('=== Partition Round-trip Test ===')
B, H, W, C, P = 2, 8, 8, 32, 4
x = torch.randn(B, H, W, C)

blocks = block_partition(x, P)
x_rec  = block_reverse(blocks, P, H, W)
print(f'Block partition:   {tuple(x.shape)} → {tuple(blocks.shape)} → {tuple(x_rec.shape)}')
print(f'  Max abs error: {(x - x_rec).abs().max():.2e}  (should be ~0)')

grids  = grid_partition(x, P)
x_rec2 = grid_reverse(grids, P, H, W)
print(f'Grid  partition:   {tuple(x.shape)} → {tuple(grids.shape)} → {tuple(x_rec2.shape)}')
print(f'  Max abs error: {(x - x_rec2).abs().max():.2e}  (should be ~0)')

print()

# ── Build MaxViT-Tiny and trace shapes ─────────────────────────────────────
print('=== MaxViT-Tiny Shape Trace ===')
backbone = build_vision_backbone('maxvit_tiny', img_size=224)
backbone.eval()
print(f'Parameters: {backbone.num_parameters():,}')
print(f'Embed dim:  {backbone.embed_dim}')
print()

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = backbone(x)

print('BackboneOutput:')
print(f'  pooled_output : {tuple(out.pooled_output.shape)}')
print(f'  feature_maps  : {[tuple(fm.shape) for fm in out.feature_maps]}')

# Demonstrate dynamic relative bias: block N=64, grid N=49 for stage-0 (56×56, P=8)
from hale_core.nn.vision.hybrid.maxvit import MaxViTBlock
stage0_block = list(backbone.stages[0].blocks.children())[0]
P_val = stage0_block.block_size
H_val = 56  # after stem
block_N = P_val ** 2
grid_N  = (H_val // P_val) ** 2
print(f'\nBlock attn: P={P_val}, window N={block_N}')
print(f'Grid  attn: H//P={H_val//P_val}, window N={grid_N}')
print(f'rel_bias table size: {stage0_block.block_attn.attn.rel_bias_table.shape}')
print(f'Block uses stored index, grid builds index dynamically (N mismatch).')


## Paper Citation & Related Work

### Primary Reference

> Z. Tu, H. Talebi, H. Zhang, F. Yang, P. Milanfar, A. Bovik, and Y. Li,
> "MaxViT: Multi-Axis Vision Transformer,"
> in *Proceedings of the European Conference on Computer Vision (ECCV)*, 2022, pp. 459–479.
> **arXiv:** https://arxiv.org/abs/2204.01697

### Related Work

| Model | Key Idea | Reference |
|-------|----------|----------|
| Swin Transformer | Hierarchical shifted-window attn | Liu et al., ICCV 2021 |
| ViL | Hierarchical global + local | Zhang et al., ICCV 2021 |
| CoAtNet | Conv stem + attention stages | Dai et al., NeurIPS 2021 |
| HiLo | Local-global attention split by freq. | Pan et al., NeurIPS 2022 |
| ViTDet | ViT backbone with windowed attn + FPN | Li et al., ECCV 2022 |

### Key Results from Paper

- **MaxViT-T:** 83.62% ImageNet top-1 at 31 M params — matches Swin-B (88 M)
- **MaxViT-L:** 86.40% at 212 M params (no extra data)
- **MaxViT-XL (JFT-pretrained):** 88.70% top-1 on ImageNet
- Scales from 224 to 512px with consistent improvement: +2.7% for MaxViT-XL
- Object detection COCO AP: 57.5 box AP with MaxViT-L backbone (ATSS detector)

### Complexity Comparison

| Method | Attention Cost |
|--------|---------------|
| ViT | $O(N^2)$ |
| Swin | $O(N \cdot P^2)$ — local only |
| MaxViT | $O(N \cdot P^2)$ — local **and** global |
| Performer / Linformer | $O(N)$ — approx, loses accuracy |
